In [1]:
import os

import sugartrail
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()

In [2]:
sugartrail.api.basic_auth.username = os.environ["COMPANIES_HOUSE_KEY"]

In [3]:
entities = [
    # {"officer_id": "uhmCAOx6PDrXSxKDXJSD1Vv2prc"},
    # {"officer_id": "3wTyHYmLN5-J6XiTww5SL0iL3fI"},
    # {"officer_id": "lBdRiCfTDhMcaLwOU6393XUfPDg"},
    # {"officer_id": "KwkjxuswE9qwWKLU0ndEaau9cq0"},
    # {"officer_id": "g8BmvnpH8blqT87i93sgJeowx7I"},
    # {"officer_id": "D-2pqWTW2QY0ooHbL5O7soMwTRc"},
    # {"officer_id": "WtiEW0LL-mMmPaRLrQSCjsWBpXY"},
    # Reform
    # {"officer_id": "iDabUesOWH_lKM0l2q6XEefh4eU"},
    # {"officer_id": "tW56blTqOI_bXYCDkDghRy3LQtU"},
    {"company_id": "08527773"},
    {"company_id": "09763501"},
    {"company_id": "11694875"},
    {"company_id": "05090691"},
    # Anti-rights
    {"company_id": "09923116"}, # ADF International (UK)
    {"company_id": "12338881"}, # LGB Alliance
    {"company_id": "05358744"}, # CROSSROADS CRISIS PREGNANCY CENTRE, HARROW
    {"company_id": "14041486"}, # WOMEN'S RIGHTS NETWORK LTD
    {"company_id": "08010183"}, # SEEN LTD
]

In [4]:
entity_graphs = []
for entity in entities: 
    if list(entity.keys())[0] == "officer_id":
        entity_graphs.append(sugartrail.base.Network(officer_id=entity["officer_id"]))
    elif list(entity.keys())[0] == "address":
        entity_graphs.append(sugartrail.base.Network(address=entity["address"]))
    elif list(entity.keys())[0] == "company_id":
        entity_graphs.append(sugartrail.base.Network(company_id=entity["company_id"]))

In [5]:
import pathlib
from pprint import pprint

for entity in entity_graphs: 
    print(f'Processing company_id={entity.company_id} officer_id={entity.officer_id} address={entity.address}')
    resp = sugartrail.api.get_accounts_changes(entity.company_id)
    for i in resp['items']:
        # print(f'Account type={i["type"]}, date={i["description_values"]["made_up_date"]}>')
        print(i["links"]["document_metadata"])
        resp2 = sugartrail.api.make_request(i["links"]["document_metadata"], entity.company_id, "company", "account")
        # print(f'AccountDocument {resp2["resources"].keys()}')
        for doc_format in resp2["resources"]:
            if doc_format == "application/pdf":
                file_ext = "pdf"
            elif doc_format == "application/xhtml+xml":
                file_ext = "xml"
            else:
                raise Exception(f"Unsupported doc format: {doc_format}")
            # print(doc_format, resp2["links"]["document"])
            print(resp2["links"]["document"])
            resp3 = sugartrail.api.make_request(resp2["links"]["document"], entity.company_id, "company", "account_content", headers={"Accept": doc_format})
            fn = f'{sugartrail.const.data_path}accounts/{entity.company_id}/{i["transaction_id"]}-{i["description_values"]["made_up_date"]}.{file_ext}'
            if pathlib.Path(fn).exists():
                continue
            # if file_ext == "xbrl":
            #     print(resp3[:100])
            pathlib.Path(fn).parent.mkdir(parents=True, exist_ok=True)
            with open(fn, 'wb') as f:
                f.write(resp3)
            print(f'Successfully wrote {fn}')
            # break
        break
    # entity.get_company_records_from_id()
    # pprint(entity.company_records)
    # break

Processing company company_id=09923116
Successfully wrote ../assets/accounts/09923116/MzQ2MDU3Mjc4MWFkaXF6a2N4-2024-06-30.pdf
b'<?xml version="1.0" encoding="utf-8" standalone="no"?><html xmlns="http://www.w3.org/1999/xhtml" xml'
Successfully wrote ../assets/accounts/09923116/MzQ2MDU3Mjc4MWFkaXF6a2N4-2024-06-30.xbrl
Processing company company_id=12338881
Successfully wrote ../assets/accounts/12338881/MzQ3ODA3Nzc1M2FkaXF6a2N4-2024-11-30.pdf
b'<?xml version="1.0" encoding="utf-8" standalone="no"?><html xmlns="http://www.w3.org/1999/xhtml" xml'
Successfully wrote ../assets/accounts/12338881/MzQ3ODA3Nzc1M2FkaXF6a2N4-2024-11-30.xbrl
Processing company company_id=05358744
Successfully wrote ../assets/accounts/05358744/MzQyNDM0NTIwOGFkaXF6a2N4-2024-02-28.pdf
b'<?xml version="1.0" encoding="UTF-8" standalone="no"?><html xmlns="http://www.w3.org/1999/xhtml" xml'
Successfully wrote ../assets/accounts/05358744/MzQyNDM0NTIwOGFkaXF6a2N4-2024-02-28.xbrl
Processing company company_id=14041486
Success